# Specific Humidity Correlation Cross-Sections

Notebook ini membangun potongan korelasi time-vertical untuk specific humidity ERA5 regional (`q`) pada wilayah `95-125E, -6 to 2`.

Output yang dibuat:
- korelasi bulanan simultan dengan Niño3.4 centered 3-bulan
- korelasi DJF-anchored dengan konvensi tahun JF
- `P1 = 1981-2006`, `P2 = 2007-2025`, dan `Δr = P2 - P1`

Konvensi DJF yang dipakai:
- `DJF 1981 = Dec 1980 + Jan 1981 + Feb 1981`
- window untuk tahun target adalah `Jul(y-1) ... Feb(y)`.


## 1. Imports and settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import pearsonr

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.titleweight': 'regular',
    'figure.dpi': 120,
})

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
Q_PATH = PROJECT_ROOT / 'external/ClimateData/era5-monthly/specific_humidity_1980-2025_1000hpa-100hpa.nc'
NINO34_PATH = PROJECT_ROOT / 'external/ClimateData/index-monthly/nino34.anom.csv'
RESULTS_DIR = PROJECT_ROOT / 'results/analisis_korelasi_26-19'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
SEASON_YEARS = np.arange(1981, 2026)

MONTH_SLOTS = [
    ('Jul(0)', 7, -1),
    ('Aug(0)', 8, -1),
    ('Sep(0)', 9, -1),
    ('Oct(0)', 10, -1),
    ('Nov(0)', 11, -1),
    ('Dec(0)', 12, -1),
    ('Jan(1)', 1, 0),
    ('Feb(1)', 2, 0),
]
SLOT_POS = np.arange(len(MONTH_SLOTS))
SLOT_LABELS = [item[0] for item in MONTH_SLOTS]
PRESSURE_TICKS = [1000, 925, 850, 700, 500, 400, 300, 200, 100]
corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)


## 2. Load data

In [ ]:
def open_q_dataset(path: Path) -> xr.Dataset:
    ds = xr.open_dataset(
        path,
        chunks={'time': 12, 'level': 1, 'lat': 181, 'lon': 360},
        decode_times=False,
    )
    ds = ds.drop_vars('step', errors='ignore')
    return xr.decode_cf(ds)


def load_nino34_series(path: Path) -> tuple[pd.Series, pd.Series]:
    df = pd.read_csv(path, parse_dates=['Date'])
    value_col = next(col for col in df.columns if col != 'Date')
    series = pd.to_numeric(df[value_col], errors='coerce').replace(-9999.0, np.nan)
    series = pd.Series(series.to_numpy(), index=df['Date'], name='nino34').sort_index()
    series = series.loc['1980-01-01':'2025-12-01'].copy()
    series_3mo = series.rolling(3, center=True, min_periods=3).mean()
    return series, series_3mo


def monthly_anomaly(da: xr.DataArray) -> xr.DataArray:
    return da.groupby('time.month') - da.groupby('time.month').mean('time')


def build_djf_index(nino_raw: pd.Series, years: np.ndarray) -> pd.Series:
    values = []
    for year in years:
        dec = pd.Timestamp(year - 1, 12, 1)
        jan = pd.Timestamp(year, 1, 1)
        feb = pd.Timestamp(year, 2, 1)
        values.append(np.nanmean([nino_raw.loc[dec], nino_raw.loc[jan], nino_raw.loc[feb]]))
    return pd.Series(values, index=years, name='nino_djf')


def season_window_date(year: int, month: int, year_shift: int) -> pd.Timestamp:
    return pd.Timestamp(year + year_shift, month, 1)


def build_window_cube(
    q_anom: xr.DataArray,
    nino_centered_3mo: pd.Series,
    nino_djf: pd.Series,
    years: np.ndarray,
) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    q_rows = []
    nino_sim_rows = []
    nino_djf_rows = []

    for year in years:
        q_slots = []
        sim_slots = []
        djf_slots = []
        for _, month, year_shift in MONTH_SLOTS:
            date = season_window_date(year, month, year_shift)
            q_slots.append(q_anom.sel(time=date).values)
            sim_slots.append(float(nino_centered_3mo.loc[date]))
            djf_slots.append(float(nino_djf.loc[year]))
        q_rows.append(np.stack(q_slots, axis=0))
        nino_sim_rows.append(np.array(sim_slots, dtype=float))
        nino_djf_rows.append(np.array(djf_slots, dtype=float))

    q_cube = xr.DataArray(
        np.stack(q_rows, axis=0),
        coords={'season_year': years, 'slot': SLOT_POS, 'level': q_anom['level'].values},
        dims=('season_year', 'slot', 'level'),
        name='q_anom_cube',
    )
    nino_sim_cube = xr.DataArray(
        np.stack(nino_sim_rows, axis=0),
        coords={'season_year': years, 'slot': SLOT_POS},
        dims=('season_year', 'slot'),
        name='nino_centered_3mo',
    )
    nino_djf_cube = xr.DataArray(
        np.stack(nino_djf_rows, axis=0),
        coords={'season_year': years, 'slot': SLOT_POS},
        dims=('season_year', 'slot'),
        name='nino_djf',
    )
    return q_cube, nino_sim_cube, nino_djf_cube


def corr_by_slot_level(q_cube: xr.DataArray, nino_cube: xr.DataArray) -> xr.DataArray:
    q_vals = q_cube.transpose('season_year', 'slot', 'level').values
    n_vals = nino_cube.transpose('season_year', 'slot').values
    out = np.full((q_vals.shape[1], q_vals.shape[2]), np.nan, dtype=float)

    for i in range(q_vals.shape[1]):
        x = n_vals[:, i]
        for j in range(q_vals.shape[2]):
            y = q_vals[:, i, j]
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.sum() >= 3 and np.nanstd(x[mask]) > 0 and np.nanstd(y[mask]) > 0:
                out[i, j] = pearsonr(x[mask], y[mask])[0]

    return xr.DataArray(
        out,
        coords={'slot': SLOT_POS, 'level': q_cube['level'].values},
        dims=('slot', 'level'),
        name='corr',
    )


def plot_cross_section(ax, da: xr.DataArray, title: str, cbar_label: str) -> None:
    mesh = ax.contourf(
        da['slot'].values,
        da['level'].values,
        da.transpose('level', 'slot').values,
        levels=corr_levels,
        cmap='bwr',
        extend='both',
    )
    ax.contour(
        da['slot'].values,
        da['level'].values,
        da.transpose('level', 'slot').values,
        levels=corr_ticks,
        colors='k',
        linewidths=0.5,
        alpha=0.45,
    )
    ax.set_title(title, loc='center', fontsize=18, fontweight='bold', pad=22)
    ax.set_xticks(SLOT_POS)
    ax.set_xticklabels(SLOT_LABELS)
    ax.set_xlabel('Bulan', fontsize=17, labelpad=10)
    ax.set_ylim(float(da['level'].max()), float(da['level'].min()))
    ax.set_yticks(da['level'].values)
    ax.set_yticklabels([str(int(v)) for v in da['level'].values])
    ax.set_ylabel('Level tekanan (hPa)', fontsize=17, labelpad=12)
    ax.tick_params(axis='both', labelsize=13)
    return mesh


In [ ]:


q_ds = open_q_dataset(Q_PATH)
q_box = q_ds['q'].sel(lat=slice(2, -6), lon=slice(95, 125))
lat_weights = np.cos(np.deg2rad(q_box['lat']))
q_region_mean = q_box.weighted(lat_weights).mean(('lat', 'lon'))
q_anom = monthly_anomaly(q_region_mean).load()

nino_raw, nino_centered_3mo = load_nino34_series(NINO34_PATH)
nino_djf = build_djf_index(nino_raw, SEASON_YEARS)

print('q dataset dims:', dict(q_ds.sizes))
print('regional mean dims:', dict(q_region_mean.sizes))
print('anomaly dims:', dict(q_anom.sizes))
print('nino range:', nino_raw.index.min(), 'to', nino_raw.index.max())
print('DJF years:', int(SEASON_YEARS.min()), 'to', int(SEASON_YEARS.max()))


## 3. Sanity check and correlation cubes

In [ ]:
print('q_anom dims:', dict(q_anom.sizes))
print('q_anom time range:', pd.Timestamp(q_anom['time'].values[0]), 'to', pd.Timestamp(q_anom['time'].values[-1]))
print('q_anom levels:', q_anom['level'].values)

q_cube_p1, nino_sim_p1, nino_djf_p1 = build_window_cube(q_anom, nino_centered_3mo, nino_djf, P1_YEARS)
q_cube_p2, nino_sim_p2, nino_djf_p2 = build_window_cube(q_anom, nino_centered_3mo, nino_djf, P2_YEARS)

corr_sim_p1 = corr_by_slot_level(q_cube_p1, nino_sim_p1)
corr_sim_p2 = corr_by_slot_level(q_cube_p2, nino_sim_p2)
corr_djf_p1 = corr_by_slot_level(q_cube_p1, nino_djf_p1)
corr_djf_p2 = corr_by_slot_level(q_cube_p2, nino_djf_p2)

corr_sim_delta = corr_sim_p2 - corr_sim_p1
corr_djf_delta = corr_djf_p2 - corr_djf_p1

print('simultaneous P1 shape:', corr_sim_p1.shape)
print('simultaneous P2 shape:', corr_sim_p2.shape)
print('DJF-anchored P1 shape:', corr_djf_p1.shape)
print('DJF-anchored P2 shape:', corr_djf_p2.shape)


## 4. Plot simultaneous monthly correlation

In [ ]:
plot_specs = [
    (corr_sim_p1, 'Korelasi Specific Humidity vs Nino34 - P1 1981-2006', 'Korelasi (r)', 'corr_specifichumidity_simultaneous_p1.png'),
    (corr_sim_p2, 'Korelasi Specific Humidity vs Nino34 - P2 2007-2025', 'Korelasi (r)', 'corr_specifichumidity_simultaneous_p2.png'),
    (corr_sim_delta, 'Selisih Korelasi Specific Humidity vs Nino34 - Δr = P2 - P1', 'Selisih korelasi', 'corr_specifichumidity_simultaneous_delta.png'),
]

for da, title, cbar_label, filename in plot_specs:
    fig, ax = plt.subplots(figsize=(9.5, 6.6))
    mesh = plot_cross_section(ax, da, title, cbar_label)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.04, shrink=0.92, ticks=corr_ticks)
    cbar.set_label(cbar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()


## 5. Plot DJF-anchored correlation

In [ ]:
plot_specs = [
    (corr_djf_p1, 'Korelasi Specific Humidity vs Nino34 DJF - P1 1981-2006', 'Korelasi (r)', 'corr_specifichumidity_djf_p1.png'),
    (corr_djf_p2, 'Korelasi Specific Humidity vs Nino34 DJF - P2 2007-2025', 'Korelasi (r)', 'corr_specifichumidity_djf_p2.png'),
    (corr_djf_delta, 'Selisih Korelasi Specific Humidity vs Nino34 DJF - Δr = P2 - P1', 'Selisih korelasi', 'corr_specifichumidity_djf_delta.png'),
]

for da, title, cbar_label, filename in plot_specs:
    fig, ax = plt.subplots(figsize=(9.5, 6.6))
    mesh = plot_cross_section(ax, da, title, cbar_label)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.04, shrink=0.92, ticks=corr_ticks)
    cbar.set_label(cbar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
